# OOD Generalization Test: Adversarial Corruption vs Baseline

Testing IRED's main hypothesis: Does adversarial corruption improve generalization to harder instances?

In [ ]:
# Setup and clone repository
!rm -rf energy-based-model
!git clone https://github.com/mdkrasnow/energy-based-model.git
%cd energy-based-model

In [ ]:
# Install dependencies
!pip install -q torch torchvision einops accelerate tqdm tabulate matplotlib numpy pandas seaborn ema-pytorch ipdb

In [ ]:
# Download datasets
!cd data && bash download-satnet.sh  # Easy sudoku data for training
!cd data && bash download-rrn.sh     # Hard sudoku data for OOD testing

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
import json
import os
import time
from datetime import datetime

# Set style
plt.style.use('default')
sns.set_palette("husl")

# GPU Setup and Detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    # Clear cache to ensure clean start
    torch.cuda.empty_cache()
else:
    print("WARNING: CUDA not available. Training will be slower on CPU.")
    
# Determine optimal number of data workers
import multiprocessing
num_workers = min(4, multiprocessing.cpu_count())
print(f"Data workers to use: {num_workers}")

In [ ]:
# GPU Performance Optimization Settings
if torch.cuda.is_available():
    # Enable TF32 for better performance on Ampere GPUs (A100, RTX 30xx)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    # Enable cudnn autotuner for optimized convolution algorithms
    torch.backends.cudnn.benchmark = True
    
    # Set environment variables for optimal GPU performance
    os.environ['CUDA_LAUNCH_BLOCKING'] = '0'  # Async CUDA operations
    
    print("GPU optimizations enabled:")
    print("  - TF32: Enabled (if supported)")
    print("  - cuDNN benchmark: Enabled")
    print("  - Async CUDA operations: Enabled")
else:
    print("Running on CPU - GPU optimizations skipped")

# Training parameters optimized for OOD testing
COMMON_ARGS = {
    'dataset': 'sudoku',  # Train on EASY data (SAT-Net)
    'model': 'sudoku',
    'batch_size': 64,  # Adjust based on GPU memory if needed
    'diffusion_steps': 100,
    'supervise_energy_landscape': 'True',
    'train_num_steps': 25000,  # Sufficient for convergence
    'cond_mask': True,         # Required for sudoku
    'save_csv_logs': True,
    'csv_log_interval': 250,
    'data_workers': num_workers  # Use multiple workers for faster data loading
}

# Adversarial corruption parameters
ADV_ARGS = {
    'use_adversarial_corruption': True,
    'anm_warmup_steps': 2500,      # 10% warmup
    'anm_adversarial_steps': 3,
    'anm_distance_penalty': 0.1
}

print("Training on EASY data (sudoku SAT-Net)")
print("Testing on HARD data (sudoku-rrn RRN)")
print(f"Total training steps: {COMMON_ARGS['train_num_steps']}")
print(f"Batch size: {COMMON_ARGS['batch_size']}")
print(f"Data workers: {COMMON_ARGS['data_workers']}")
print(f"Device: {device}")

In [ ]:
# Training parameters optimized for OOD testing
COMMON_ARGS = {
    'dataset': 'sudoku',  # Train on EASY data (SAT-Net)
    'model': 'sudoku',
    'batch_size': 64,
    'diffusion_steps': 100,
    'supervise_energy_landscape': 'True',
    'train_num_steps': 25000,  # Sufficient for convergence
    'cond_mask': True,         # Required for sudoku
    'save_csv_logs': True,
    'csv_log_interval': 250
}

# Adversarial corruption parameters
ADV_ARGS = {
    'use_adversarial_corruption': True,
    'anm_warmup_steps': 2500,      # 10% warmup
    'anm_adversarial_steps': 3,
    'anm_distance_penalty': 0.1
}

print("Training on EASY data (sudoku SAT-Net)")
print("Testing on HARD data (sudoku-rrn RRN)")
print(f"Total training steps: {COMMON_ARGS['train_num_steps']}")

print("=== STARTING BASELINE TRAINING ===")
start_time = time.time()

# Add environment variable for GPU if available
env_prefix = ""
if torch.cuda.is_available():
    env_prefix = "CUDA_VISIBLE_DEVICES=0 "
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

baseline_cmd = f"""{env_prefix}python train.py \
    --dataset {COMMON_ARGS['dataset']} \
    --model {COMMON_ARGS['model']} \
    --cond_mask True \
    --batch_size {COMMON_ARGS['batch_size']} \
    --diffusion_steps {COMMON_ARGS['diffusion_steps']} \
    --supervise-energy-landscape {COMMON_ARGS['supervise_energy_landscape']} \
    --train-num-steps {COMMON_ARGS['train_num_steps']} \
    --data-workers {COMMON_ARGS['data_workers']} \
    --save-csv-logs \
    --csv-log-interval {COMMON_ARGS['csv_log_interval']} \
    --csv-log-dir ./csv_logs_baseline_ood
"""

print(f"Running with {COMMON_ARGS['data_workers']} data workers")
!{baseline_cmd}

baseline_time = time.time() - start_time
print(f"\nBaseline training completed in {baseline_time:.1f} seconds ({baseline_time/60:.1f} minutes)")

In [ ]:
print("=== STARTING BASELINE TRAINING ===")
start_time = time.time()

baseline_cmd = f"""python train.py \
    --dataset {COMMON_ARGS['dataset']} \
    --model {COMMON_ARGS['model']} \
    --cond_mask True \
    --batch_size {COMMON_ARGS['batch_size']} \
    --diffusion_steps {COMMON_ARGS['diffusion_steps']} \
    --supervise-energy-landscape {COMMON_ARGS['supervise_energy_landscape']} \
    --train-num-steps {COMMON_ARGS['train_num_steps']} \
    --save-csv-logs \
    --csv-log-interval {COMMON_ARGS['csv_log_interval']} \
    --csv-log-dir ./csv_logs_baseline_ood
"""

!{baseline_cmd}

baseline_time = time.time() - start_time
print(f"\nBaseline training completed in {baseline_time:.1f} seconds")

print("=== STARTING ADVERSARIAL CORRUPTION TRAINING ===")
start_time = time.time()

# Add environment variable for GPU if available
env_prefix = ""
if torch.cuda.is_available():
    env_prefix = "CUDA_VISIBLE_DEVICES=0 "
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

adversarial_cmd = f"""{env_prefix}python train.py \
    --dataset {COMMON_ARGS['dataset']} \
    --model {COMMON_ARGS['model']} \
    --cond_mask True \
    --batch_size {COMMON_ARGS['batch_size']} \
    --diffusion_steps {COMMON_ARGS['diffusion_steps']} \
    --supervise-energy-landscape {COMMON_ARGS['supervise_energy_landscape']} \
    --train-num-steps {COMMON_ARGS['train_num_steps']} \
    --data-workers {COMMON_ARGS['data_workers']} \
    --save-csv-logs \
    --csv-log-interval {COMMON_ARGS['csv_log_interval']} \
    --csv-log-dir ./csv_logs_adversarial_ood \
    --use-adversarial-corruption {ADV_ARGS['use_adversarial_corruption']} \
    --anm-warmup-steps {ADV_ARGS['anm_warmup_steps']} \
    --anm-adversarial-steps {ADV_ARGS['anm_adversarial_steps']} \
    --anm-distance-penalty {ADV_ARGS['anm_distance_penalty']}
"""

print(f"Running with {COMMON_ARGS['data_workers']} data workers")
!{adversarial_cmd}

adversarial_time = time.time() - start_time
print(f"\nAdversarial training completed in {adversarial_time:.1f} seconds ({adversarial_time/60:.1f} minutes)")

In [ ]:
print("=== STARTING ADVERSARIAL CORRUPTION TRAINING ===")
start_time = time.time()

adversarial_cmd = f"""python train.py \
    --dataset {COMMON_ARGS['dataset']} \
    --model {COMMON_ARGS['model']} \
    --cond_mask True \
    --batch_size {COMMON_ARGS['batch_size']} \
    --diffusion_steps {COMMON_ARGS['diffusion_steps']} \
    --supervise-energy-landscape {COMMON_ARGS['supervise_energy_landscape']} \
    --train-num-steps {COMMON_ARGS['train_num_steps']} \
    --save-csv-logs \
    --csv-log-interval {COMMON_ARGS['csv_log_interval']} \
    --csv-log-dir ./csv_logs_adversarial_ood \
    --use-adversarial-corruption {ADV_ARGS['use_adversarial_corruption']} \
    --anm-warmup-steps {ADV_ARGS['anm_warmup_steps']} \
    --anm-adversarial-steps {ADV_ARGS['anm_adversarial_steps']} \
    --anm-distance-penalty {ADV_ARGS['anm_distance_penalty']}
"""

!{adversarial_cmd}

adversarial_time = time.time() - start_time
print(f"\nAdversarial training completed in {adversarial_time:.1f} seconds")

## OOD Evaluation Setup

In [ ]:
def load_latest_csv(csv_dir, pattern):
    """Load the most recent CSV file matching pattern"""
    csv_path = Path(csv_dir)
    if not csv_path.exists():
        print(f"Directory {csv_dir} does not exist")
        return None
    
    files = list(csv_path.glob(pattern))
    if not files:
        print(f"No files found matching {pattern} in {csv_dir}")
        return None
        
    latest_file = max(files, key=lambda x: x.stat().st_mtime)
    print(f"Loading: {latest_file}")
    return pd.read_csv(latest_file)

def get_final_metrics(df_val, metric_name):
    """Extract final performance for a specific metric"""
    if df_val is None or len(df_val) == 0:
        return None
    
    metric_data = df_val[df_val['metric_name'] == metric_name]
    if len(metric_data) == 0:
        return None
        
    # Get final value
    return metric_data['metric_value'].iloc[-1]

def calculate_convergence_speed(df_val, metric_name, target_accuracy=0.8):
    """Calculate steps to reach target accuracy"""
    if df_val is None:
        return None
        
    metric_data = df_val[df_val['metric_name'] == metric_name]
    if len(metric_data) == 0:
        return None
        
    # Find first step where accuracy >= target
    target_steps = metric_data[metric_data['metric_value'] >= target_accuracy]
    if len(target_steps) == 0:
        return None  # Never reached target
        
    return target_steps['step'].iloc[0]

print("OOD evaluation utilities loaded")

In [ ]:
# Load training results
print("Loading baseline results...")
baseline_train = load_latest_csv('./csv_logs_baseline_ood', 'training_metrics_*.csv')
baseline_val = load_latest_csv('./csv_logs_baseline_ood', 'validation_metrics_*.csv')

print("\nLoading adversarial results...")
adversarial_train = load_latest_csv('./csv_logs_adversarial_ood', 'training_metrics_*.csv')
adversarial_val = load_latest_csv('./csv_logs_adversarial_ood', 'validation_metrics_*.csv')
adversarial_energy = load_latest_csv('./csv_logs_adversarial_ood', 'energy_metrics_*.csv')

print("\nData loading complete.")

## IRED Hypothesis Testing: Key Metrics Analysis

In [ ]:
# Extract key metrics for IRED hypothesis testing
print("=== IRED HYPOTHESIS TESTING METRICS ===")

# Sudoku metrics to analyze
SUDOKU_METRICS = ['accuracy', 'consistency', 'board_accuracy']

# Initialize results dictionary
results = {
    'baseline': {'easy': {}, 'hard': {}},
    'adversarial': {'easy': {}, 'hard': {}}
}

# Extract metrics for each method and dataset
for metric in SUDOKU_METRICS:
    print(f"\nAnalyzing {metric}:")
    
    # Baseline metrics
    if baseline_val is not None:
        # Easy data (in-distribution validation)
        baseline_easy = get_final_metrics(baseline_val[baseline_val['prefix'] == 'train_sample'], metric)
        # Hard data (OOD test - sudoku-rrn-test)
        baseline_hard = get_final_metrics(baseline_val[baseline_val['prefix'] == 'sudoku-rrn-test'], metric)
        
        results['baseline']['easy'][metric] = baseline_easy
        results['baseline']['hard'][metric] = baseline_hard
        
        print(f"  Baseline - Easy: {baseline_easy:.4f}, Hard: {baseline_hard:.4f}")
        if baseline_easy and baseline_hard:
            gap_baseline = baseline_easy - baseline_hard
            print(f"  Baseline Generalization Gap: {gap_baseline:.4f}")
    
    # Adversarial metrics
    if adversarial_val is not None:
        # Easy data (in-distribution validation)
        adversarial_easy = get_final_metrics(adversarial_val[adversarial_val['prefix'] == 'train_sample'], metric)
        # Hard data (OOD test - sudoku-rrn-test)
        adversarial_hard = get_final_metrics(adversarial_val[adversarial_val['prefix'] == 'sudoku-rrn-test'], metric)
        
        results['adversarial']['easy'][metric] = adversarial_easy
        results['adversarial']['hard'][metric] = adversarial_hard
        
        print(f"  Adversarial - Easy: {adversarial_easy:.4f}, Hard: {adversarial_hard:.4f}")
        if adversarial_easy and adversarial_hard:
            gap_adversarial = adversarial_easy - adversarial_hard
            print(f"  Adversarial Generalization Gap: {gap_adversarial:.4f}")
            
            # Calculate improvement
            if baseline_easy and baseline_hard:
                gap_improvement = gap_baseline - gap_adversarial
                print(f"  Gap Improvement: {gap_improvement:.4f} ({'better' if gap_improvement > 0 else 'worse'})")

print("\nMetrics extraction complete.")

In [ ]:
# Convergence Speed Analysis
print("\n=== CONVERGENCE SPEED ANALYSIS ===")

convergence_results = {}
TARGET_ACCURACY = 0.7  # 70% accuracy target

for metric in ['accuracy', 'board_accuracy']:
    print(f"\nConvergence analysis for {metric} (target: {TARGET_ACCURACY})")
    
    # Baseline convergence
    if baseline_val is not None:
        baseline_conv_easy = calculate_convergence_speed(
            baseline_val[baseline_val['prefix'] == 'train_sample'], metric, TARGET_ACCURACY
        )
        baseline_conv_hard = calculate_convergence_speed(
            baseline_val[baseline_val['prefix'] == 'sudoku-rrn-test'], metric, TARGET_ACCURACY
        )
        
        print(f"  Baseline - Easy: {baseline_conv_easy} steps, Hard: {baseline_conv_hard} steps")
    
    # Adversarial convergence
    if adversarial_val is not None:
        adversarial_conv_easy = calculate_convergence_speed(
            adversarial_val[adversarial_val['prefix'] == 'train_sample'], metric, TARGET_ACCURACY
        )
        adversarial_conv_hard = calculate_convergence_speed(
            adversarial_val[adversarial_val['prefix'] == 'sudoku-rrn-test'], metric, TARGET_ACCURACY
        )
        
        print(f"  Adversarial - Easy: {adversarial_conv_easy} steps, Hard: {adversarial_conv_hard} steps")
        
        # Speed improvement calculation
        if baseline_conv_hard and adversarial_conv_hard:
            speed_improvement = baseline_conv_hard - adversarial_conv_hard
            print(f"  Speed improvement on hard data: {speed_improvement} steps")
            if speed_improvement > 0:
                print(f"  Adversarial converges {speed_improvement} steps faster!")
    
    convergence_results[metric] = {
        'baseline_easy': baseline_conv_easy,
        'baseline_hard': baseline_conv_hard,
        'adversarial_easy': adversarial_conv_easy,
        'adversarial_hard': adversarial_conv_hard
    }

## Comprehensive Results Visualization

In [ ]:
# Create comprehensive visualization
fig = plt.figure(figsize=(20, 15))

# 1. Training Loss Comparison (2x2 subplot)
ax1 = plt.subplot(3, 3, 1)
if baseline_train is not None and adversarial_train is not None:
    plt.plot(baseline_train['step'], baseline_train['total_loss'], 'b-', alpha=0.7, label='Baseline')
    plt.plot(adversarial_train['step'], adversarial_train['total_loss'], 'r-', alpha=0.7, label='Adversarial')
    plt.title('Total Loss During Training')
    plt.xlabel('Training Step')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

# 2. Energy Loss Comparison
ax2 = plt.subplot(3, 3, 2)
if baseline_train is not None and adversarial_train is not None:
    plt.plot(baseline_train['step'], baseline_train['loss_energy'], 'b-', alpha=0.7, label='Baseline')
    plt.plot(adversarial_train['step'], adversarial_train['loss_energy'], 'r-', alpha=0.7, label='Adversarial')
    plt.title('Energy Loss During Training')
    plt.xlabel('Training Step')
    plt.ylabel('Energy Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

# 3. Curriculum Weight Evolution
ax3 = plt.subplot(3, 3, 3)
if adversarial_energy is not None:
    plt.plot(adversarial_energy['step'], adversarial_energy['curriculum_weight'], 'g-', alpha=0.7)
    plt.title('Adversarial Curriculum Weight')
    plt.xlabel('Training Step')
    plt.ylabel('Weight')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1.1)

# 4. Generalization Gap Comparison (Key Metric)
ax4 = plt.subplot(3, 3, 4)
metrics = ['accuracy', 'consistency', 'board_accuracy']
baseline_gaps = []
adversarial_gaps = []

for metric in metrics:
    if (results['baseline']['easy'].get(metric) and results['baseline']['hard'].get(metric) and 
        results['adversarial']['easy'].get(metric) and results['adversarial']['hard'].get(metric)):
        
        baseline_gap = results['baseline']['easy'][metric] - results['baseline']['hard'][metric]
        adversarial_gap = results['adversarial']['easy'][metric] - results['adversarial']['hard'][metric]
        
        baseline_gaps.append(baseline_gap)
        adversarial_gaps.append(adversarial_gap)
    else:
        baseline_gaps.append(0)
        adversarial_gaps.append(0)

x = np.arange(len(metrics))
width = 0.35

plt.bar(x - width/2, baseline_gaps, width, label='Baseline', alpha=0.7, color='blue')
plt.bar(x + width/2, adversarial_gaps, width, label='Adversarial', alpha=0.7, color='red')
plt.title('Generalization Gap (Easy - Hard)')
plt.xlabel('Metric')
plt.ylabel('Gap (Lower = Better)')
plt.xticks(x, metrics, rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)

# 5. Performance on Hard Data (OOD)
ax5 = plt.subplot(3, 3, 5)
baseline_hard_vals = [results['baseline']['hard'].get(m, 0) for m in metrics]
adversarial_hard_vals = [results['adversarial']['hard'].get(m, 0) for m in metrics]

plt.bar(x - width/2, baseline_hard_vals, width, label='Baseline', alpha=0.7, color='blue')
plt.bar(x + width/2, adversarial_hard_vals, width, label='Adversarial', alpha=0.7, color='red')
plt.title('Performance on Hard Data (OOD)')
plt.xlabel('Metric')
plt.ylabel('Performance (Higher = Better)')
plt.xticks(x, metrics, rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)

# 6. Performance on Easy Data (ID)
ax6 = plt.subplot(3, 3, 6)
baseline_easy_vals = [results['baseline']['easy'].get(m, 0) for m in metrics]
adversarial_easy_vals = [results['adversarial']['easy'].get(m, 0) for m in metrics]

plt.bar(x - width/2, baseline_easy_vals, width, label='Baseline', alpha=0.7, color='blue')
plt.bar(x + width/2, adversarial_easy_vals, width, label='Adversarial', alpha=0.7, color='red')
plt.title('Performance on Easy Data (In-Distribution)')
plt.xlabel('Metric')
plt.ylabel('Performance (Higher = Better)')
plt.xticks(x, metrics, rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)

# 7. Validation Accuracy Over Time (Easy Data)
ax7 = plt.subplot(3, 3, 7)
if baseline_val is not None and adversarial_val is not None:
    baseline_easy_acc = baseline_val[
        (baseline_val['prefix'] == 'train_sample') & 
        (baseline_val['metric_name'] == 'accuracy')
    ]
    adversarial_easy_acc = adversarial_val[
        (adversarial_val['prefix'] == 'train_sample') & 
        (adversarial_val['metric_name'] == 'accuracy')
    ]
    
    if len(baseline_easy_acc) > 0:
        plt.plot(baseline_easy_acc['step'], baseline_easy_acc['metric_value'], 
                'b-', alpha=0.7, label='Baseline')
    if len(adversarial_easy_acc) > 0:
        plt.plot(adversarial_easy_acc['step'], adversarial_easy_acc['metric_value'], 
                'r-', alpha=0.7, label='Adversarial')
    
    plt.title('Accuracy on Easy Data Over Time')
    plt.xlabel('Training Step')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

# 8. Validation Accuracy Over Time (Hard Data - OOD)
ax8 = plt.subplot(3, 3, 8)
if baseline_val is not None and adversarial_val is not None:
    baseline_hard_acc = baseline_val[
        (baseline_val['prefix'] == 'sudoku-rrn-test') & 
        (baseline_val['metric_name'] == 'accuracy')
    ]
    adversarial_hard_acc = adversarial_val[
        (adversarial_val['prefix'] == 'sudoku-rrn-test') & 
        (adversarial_val['metric_name'] == 'accuracy')
    ]
    
    if len(baseline_hard_acc) > 0:
        plt.plot(baseline_hard_acc['step'], baseline_hard_acc['metric_value'], 
                'b-', alpha=0.7, label='Baseline')
    if len(adversarial_hard_acc) > 0:
        plt.plot(adversarial_hard_acc['step'], adversarial_hard_acc['metric_value'], 
                'r-', alpha=0.7, label='Adversarial')
    
    plt.title('Accuracy on Hard Data (OOD) Over Time')
    plt.xlabel('Training Step')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)

# 9. Corruption Type Usage (Adversarial Only)
ax9 = plt.subplot(3, 3, 9)
if adversarial_energy is not None and 'corruption_type' in adversarial_energy.columns:
    corruption_counts = adversarial_energy['corruption_type'].value_counts()
    plt.pie(corruption_counts.values, labels=corruption_counts.index, autopct='%1.1f%%')
    plt.title('Corruption Type Usage')
else:
    plt.text(0.5, 0.5, 'No corruption\ntype data', ha='center', va='center', transform=plt.gca().transAxes)
    plt.title('Corruption Type Usage')

plt.tight_layout()
plt.show()

## Final Results Summary

In [ ]:
print("="*80)
print("IRED HYPOTHESIS TESTING RESULTS")
print("="*80)

print("\n🎯 CORE HYPOTHESIS: Does adversarial corruption improve generalization to harder instances?")

# Summary statistics
print("\n📊 TRAINING SUMMARY:")
print(f"   Training Dataset: {COMMON_ARGS['dataset']} (SAT-Net - Easy)")
print(f"   OOD Test Dataset: sudoku-rrn (RRN - Hard)")
print(f"   Training Steps: {COMMON_ARGS['train_num_steps']:,}")
print(f"   Baseline Training Time: {baseline_time:.1f}s")
print(f"   Adversarial Training Time: {adversarial_time:.1f}s")

# Key Metrics Analysis
print("\n📈 KEY METRICS ANALYSIS:")

print("\n1️⃣ GENERALIZATION GAP (Easy - Hard Performance):")
total_improvements = 0
total_metrics = 0

for metric in SUDOKU_METRICS:
    baseline_easy = results['baseline']['easy'].get(metric)
    baseline_hard = results['baseline']['hard'].get(metric)
    adversarial_easy = results['adversarial']['easy'].get(metric)
    adversarial_hard = results['adversarial']['hard'].get(metric)
    
    if all([baseline_easy, baseline_hard, adversarial_easy, adversarial_hard]):
        baseline_gap = baseline_easy - baseline_hard
        adversarial_gap = adversarial_easy - adversarial_hard
        improvement = baseline_gap - adversarial_gap
        
        print(f"   {metric.upper()}:")
        print(f"     Baseline Gap:    {baseline_gap:.4f}")
        print(f"     Adversarial Gap: {adversarial_gap:.4f}")
        print(f"     Improvement:     {improvement:.4f} {'✅' if improvement > 0 else '❌'}")
        
        if improvement > 0:
            total_improvements += 1
        total_metrics += 1

print("\n2️⃣ OUT-OF-DISTRIBUTION PERFORMANCE (Hard Data):")
for metric in SUDOKU_METRICS:
    baseline_hard = results['baseline']['hard'].get(metric)
    adversarial_hard = results['adversarial']['hard'].get(metric)
    
    if baseline_hard and adversarial_hard:
        improvement = adversarial_hard - baseline_hard
        print(f"   {metric.upper()}: Baseline={baseline_hard:.4f}, Adversarial={adversarial_hard:.4f}, "
              f"Δ={improvement:+.4f} {'✅' if improvement > 0 else '❌'}")

print("\n3️⃣ CONVERGENCE SPEED (Steps to reach 70% accuracy):")
for metric in ['accuracy', 'board_accuracy']:
    if metric in convergence_results:
        baseline_hard = convergence_results[metric]['baseline_hard']
        adversarial_hard = convergence_results[metric]['adversarial_hard']
        
        print(f"   {metric.upper()} (Hard Data):")
        if baseline_hard:
            print(f"     Baseline: {baseline_hard:,} steps")
        else:
            print(f"     Baseline: Never reached 70%")
            
        if adversarial_hard:
            print(f"     Adversarial: {adversarial_hard:,} steps")
        else:
            print(f"     Adversarial: Never reached 70%")
            
        if baseline_hard and adversarial_hard:
            speed_improvement = baseline_hard - adversarial_hard
            print(f"     Speed improvement: {speed_improvement:,} steps {'✅' if speed_improvement > 0 else '❌'}")

# Adversarial Corruption Analysis
if adversarial_energy is not None:
    print("\n4️⃣ ADVERSARIAL CORRUPTION ANALYSIS:")
    max_curriculum = adversarial_energy['curriculum_weight'].max()
    print(f"   Max Curriculum Weight: {max_curriculum:.3f}")
    
    if 'corruption_type' in adversarial_energy.columns:
        corruption_counts = adversarial_energy['corruption_type'].value_counts()
        print("   Corruption Type Usage:")
        for corruption_type, count in corruption_counts.items():
            percentage = (count / len(adversarial_energy)) * 100
            print(f"     {corruption_type}: {count:,} ({percentage:.1f}%)")

# Final Verdict
print("\n🏆 FINAL VERDICT:")
if total_metrics > 0:
    success_rate = (total_improvements / total_metrics) * 100
    print(f"   Generalization improvements: {total_improvements}/{total_metrics} metrics ({success_rate:.1f}%)")
    
    if success_rate >= 66.7:  # 2/3 or better
        print("   ✅ HYPOTHESIS CONFIRMED: Adversarial corruption improves generalization!")
    elif success_rate >= 33.3:  # 1/3 to 2/3
        print("   ⚠️  HYPOTHESIS PARTIALLY CONFIRMED: Mixed results")
    else:
        print("   ❌ HYPOTHESIS NOT CONFIRMED: Limited improvement")
else:
    print("   ⚠️  INSUFFICIENT DATA: Unable to evaluate hypothesis")

print("\n" + "="*80)
print("Analysis complete. Check visualizations above for detailed insights.")
print("="*80)

In [ ]:
# Save results to JSON for future reference
final_results = {
    'experiment_config': {
        'training_dataset': COMMON_ARGS['dataset'],
        'ood_dataset': 'sudoku-rrn',
        'training_steps': COMMON_ARGS['train_num_steps'],
        'adversarial_config': ADV_ARGS,
        'timestamp': datetime.now().isoformat()
    },
    'performance_metrics': results,
    'convergence_analysis': convergence_results,
    'training_times': {
        'baseline_seconds': baseline_time,
        'adversarial_seconds': adversarial_time
    }
}

# Save to file
with open('ood_generalization_results.json', 'w') as f:
    json.dump(final_results, f, indent=2, default=str)

print("Results saved to ood_generalization_results.json")
print("\nExperiment completed successfully! 🎉")